# **Parcial 2 : Análisis de datos del mercado laboral GEIH 2026: DESARROLLO**

### 1.Para el total nacional calcule (a Junio de 2026): 
### - población en edad de trabajar  
### - población que no está en edad de trabajar 
### - población económicamente activa 
### - población económicamente inactiva 
### - tasa de ocupación -tasa de desempleo  

Lo primero que vamos a hacer es cargar todas las tablas que necesitamos desde el catálogo: fuerza de trabajo, características generales (cgse), ocupados y desempleados. Las pasamos a pandas para poder trabajar con ellas.

Algo importante: estas tablas ya traen todo el mes de junio de 2026, así que no necesitamos filtrar por período, trabajamos con las tablas completas.

Otra cosa que tenemos que tener presente es que cada fila trae un factor de expansión (FEX_C18), que nos dice a cuántas personas de la población representa esa fila. Por eso para obtener los totales no contamos filas, sumamos ese factor.

In [0]:
import pandas as pd

# Cargamos las tablas y las pasamos a pandas
fuerzadetrabajo = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo").toPandas()
cgse = spark.read.format("delta").table("workspace.electiva.CGSE").toPandas()
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados").toPandas()


**Población en edad de trabajar (PET)**

Primero vamos a calcular la población en edad de trabajar. La PET son las personas de 15 años o más, y en la tabla de fuerza de trabajo ya viene marcada con la variable PET: es igual a 1 cuando la persona está en edad de trabajar.

Entonces filtramos las filas donde PET == 1 y sumamos el factor de expansión FEX_C18. Sumamos el factor y no contamos filas, porque cada fila representa a muchas personas de la población, no a una sola.

Para comprobar que el resultado está bien, lo vamos a calcular de otra forma: desde la tabla de características generales (cgse), sumando el factor de las personas con P6040 >= 15 (la edad). Los dos resultados deberían ser iguales.

In [0]:
# Calcular la población en edad de trabajar (PET) desde la tabla de fuerza de trabajo
PET = fuerzadetrabajo.loc[
    fuerzadetrabajo["PET"] == 1, "FEX_C18"
].sum()

# Comprobación: PET desde características generales (personas de 15 años o más)
PET_CGSE = cgse.loc[
    cgse["P6040"] >= 15, "FEX_C18"
].sum()

print(f"Población en edad de trabajar (fuerza de trabajo): {PET:,.0f} personas")
print(f"Población en edad de trabajar (características generales): {PET_CGSE:,.0f} personas")


_**R/ A junio de 2026, la población en edad de trabajar en Colombia es de 41,202,631 personas**_

**Población que no está en edad de trabajar**

Ahora vamos a calcular la población que no está en edad de trabajar, que son las personas menores de 15 años. Para esto usamos la tabla de características generales (cgse), que trae la edad de cada persona en la variable P6040.

Filtramos las personas con P6040 < 15 y sumamos el factor de expansión FEX_C18, igual que hicimos con la PET.

Para comprobar que todo cuadra, también vamos a sumar la PET con esta población: el resultado tiene que ser la población total del país, que sale de sumar el factor de expansión de toda la tabla.

In [0]:
# Población que no está en edad de trabajar: personas menores de 15 años
poblacion_no_PET = cgse.loc[
    cgse["P6040"] < 15, "FEX_C18"
].sum()

# Comprobación: PET + no PET debe ser igual a la población total
poblacion_total = cgse["FEX_C18"].sum()

print(f"Población que no está en edad de trabajar: {poblacion_no_PET:,.0f} personas")
print(f"PET + no PET: {PET + poblacion_no_PET:,.0f} personas")
print(f"Población total: {poblacion_total:,.0f} personas")

_**R/ A junio de 2026, la población que no está en edad de trabajar en Colombia (menores de 15 años) es de 11,346,181 personas.**_

Si la sumamos con la población en edad de trabajar (41,202,631), obtenemos 52,548,812 personas, que es la población total del país y coincide con la suma del factor de expansión de toda la tabla, lo que confirma que los cálculos están bien.

**Población económicamente activa (PEA)**

Ahora vamos a calcular la población económicamente activa. La PEA son las personas en edad de trabajar que tienen empleo o que están buscando uno. En la tabla de fuerza de trabajo ya viene marcada con la variable FT (fuerza de trabajo): es igual a 1 cuando la persona hace parte de la PEA.

Filtramos las filas donde FT == 1 y sumamos el factor de expansión FEX_C18.

In [0]:
# Población económicamente activa (PEA)
PEA = fuerzadetrabajo.loc[
    fuerzadetrabajo["FT"] == 1, "FEX_C18"
].sum()

print(f"Población económicamente activa: {PEA:,.0f} personas")


_**R/ A junio de 2026, la población económicamente activa en Colombia es de 26,593,429 personas.**_

**Población económicamente inactiva (PEI)**

Ahora vamos a calcular la población económicamente inactiva. La PEI son las personas que están en edad de trabajar pero no tienen empleo ni lo están buscando (por ejemplo estudiantes, personas dedicadas a oficios del hogar o pensionados). En la tabla de fuerza de trabajo viene marcada con la variable FFT (fuera de la fuerza de trabajo): es igual a 1 cuando la persona hace parte de la PEI.

Filtramos las filas donde FFT == 1 y sumamos el factor de expansión FEX_C18.

In [0]:
# Población económicamente inactiva (PEI)
PEI = fuerzadetrabajo.loc[
    fuerzadetrabajo["FFT"] == 1, "FEX_C18"
].sum()

print(f"Población económicamente inactiva: {PEI:,.0f} personas")


_**R/ A junio de 2026, la población económicamente inactiva en Colombia es de 14,609,202 personas.**_

**Tasa de ocupación (TO)**

Ahora vamos a calcular la tasa de ocupación, que nos dice qué porcentaje de la población en edad de trabajar tiene empleo. La fórmula es:

TO = ocupados / PET × 100

Para esto necesitamos las dos cantidades: la PET, que ya la calculamos, y los ocupados. Los ocupados salen de la tabla de ocupados, sumando el factor de expansión FEX_C18, porque esa tabla solo contiene a las personas que tienen empleo.

In [0]:
# Población ocupada (PO): todas las filas de la tabla de ocupados
PO = ocupados["FEX_C18"].sum()

# Tasa de ocupación (TO)
TO = PO / PET * 100

print(f"Población ocupada: {PO:,.0f} personas")
print(f"Tasa de ocupación: {TO:.2f} %")

_**R/ A junio de 2026, la tasa de ocupación en Colombia es de 59,36 %, con 24,458,794 personas ocupadas sobre 41,202,631 en edad de trabajar.**_

**Tasa de desempleo (TD)**

Por último vamos a calcular la tasa de desempleo, que nos dice qué porcentaje de la población económicamente activa no tiene empleo pero lo está buscando. La fórmula es:

TD = desempleados / PEA × 100

A diferencia de la tasa de ocupación, aquí no dividimos entre la PET sino entre la PEA, porque el desempleo se mide solo sobre las personas que están en el mercado laboral. Los desempleados salen de la tabla de desempleados, sumando el factor de expansión FEX_C18 de las personas con DSI == 1, y la PEA ya la calculamos antes.

In [0]:
# Población desempleada (D): personas con DSI == 1 en la tabla de desempleados
D = desempleados.loc[desempleados["DSI"] == 1, "FEX_C18"].sum()

# Tasa de desempleo (TD)
TD = D / PEA * 100

print(f"Población desempleada: {D:,.0f} personas")
print(f"Población económicamente activa: {PEA:,.0f} personas")
print(f"Tasa de desempleo: {TD:.2f} %")

_**R/ A junio de 2026, la tasa de desempleo en Colombia es de 8,03 %, con 2,134,635 personas desempleadas sobre una población económicamente activa de 26,593,429.**_

### 2. Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?

Para calcular la tasa de desempleo por departamento y por sexo, usamos la misma fórmula del punto 1: TD = desempleados / PEA × 100, pero ahora calculada para cada grupo (hombres y mujeres de cada departamento).

Primero cargamos las tablas y las pasamos a pandas. Hay un detalle: la variable del sexo (P3271) está en la tabla de características generales (cgse), pero no está en la de desempleados ni en la de fuerza de trabajo. Entonces necesitamos traerla con un merge, usando como llaves DIRECTORIO, SECUENCIA_P y ORDEN, que juntas identifican a cada persona. Usamos how='left' para conservar todas las filas de la tabla original y solo agregarles el sexo (y la edad, P6040, que nos va a servir en los siguientes puntos).

In [0]:
import pandas as pd

In [0]:
#Leyendo la tabla de caracteristicas generales
ft = spark.read.format("delta").table("workspace.electiva.fuerzadetrabajo")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')


In [0]:
desempleados

Ahora agrupamos con groupby por sexo y departamento, y sumamos el factor de expansión FEX_C18. Hacemos esto dos veces: con los desempleados (DSI == 1) y con la PEA (FT == 1). Cada resultado es una serie con la cantidad de personas de cada grupo.

In [0]:
# desempleados_dpto -> cantidad de hombres y mujeres desempleados por departamento
desempleados_dpto_genero = desempleados[desempleados['DSI']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_dpto_genero

In [0]:
#cantidad de hombres y mujeres economicamente activos por departamento
pea_dpto_genero = ft[ft['FT']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

In [0]:
pea_dpto_genero

Juntamos las dos series en una sola tabla (PEA y DESEMPLEADOS) y calculamos la tasa de desempleo de cada grupo dividiendo los desempleados entre la PEA y multiplicando por 100.

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero, 'DESEMPLEADOS':desempleados_dpto_genero})

In [0]:
tabla_desempleo

In [0]:
tabla_desempleo['TD']= (tabla_desempleo['DESEMPLEADOS']/tabla_desempleo['PEA'])*100

In [0]:
tabla_desempleo.head(100)

Los departamentos y el sexo vienen como códigos numéricos, así que usamos dos diccionarios para pasarlos a nombres (por ejemplo, 63 es Quindío y 2 es Mujer). Con reset_index() convertimos los índices del groupby en columnas para poder mapearlos.

In [0]:

# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

sexo_nombre = {1:'Hombre',2:'Mujer'}

tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo_nombre)

     

In [0]:
tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo

Por último ordenamos la tabla por sexo y por tasa de desempleo, de mayor a menor. Como el orden es descendente, las mujeres quedan primero y, dentro de cada sexo, el departamento con la tasa más alta queda arriba y el de la más baja al final.

In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=[ 'Sexo','TD'], ascending=False)
tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo']=='Hombre']

_**R/ A junio de 2026, los departamentos con mayor y menor tasa de desempleo son:**_

- Mujeres: la mayor tasa está en Putumayo (25,02 %), seguido de Sucre (20,49 %) y Vichada (18,44 %). La menor está en Vaupés (1,78 %), seguido de Amazonas (2,61 %) y Huila (6,58 %).
- Hombres: la mayor tasa está en San Andrés (20,98 %), seguido de Putumayo (19,68 %) y Arauca (14,49 %). La menor está en Vaupés (3,00 %), seguido de Cauca (3,05 %) y Risaralda (4,00 %).


### 3.  Tomando en consideración únicamente los siguientes grupos de edad:
### 
### - 18 a 24 
### - 24 a 30 
### - 30 a 40 
### - 40 a 50  
### - 60 - en adelante 
### 
### Para cada uno de los departamentos, cual es el grupo de edad con mayor y menor 
### tasa de desempleo ?

Ahora vamos a calcular la tasa de desempleo por departamento y por grupo de edad, con la misma fórmula de antes: TD = desempleados / PEA × 100. Para esto no necesitamos volver a traer la edad, porque en el punto 2 ya anexamos P6040 a las tablas ft y desempleados.

Primero creamos los grupos de edad con pd.cut. Los rangos del enunciado se cruzan en los extremos (24 y 30) y falta el de 50 a 60, entonces usamos los límites incluyendo el inferior y excluyendo el superior: 18 a 23, 24 a 29, 30 a 39, 40 a 49 y 60 en adelante. Creamos también el grupo de 50 a 60 solo para poder separarlo y descartarlo después, porque el enunciado dice “únicamente” esos grupos.

In [0]:
# Grupos de edad (incluye el límite inferior, excluye el superior)
bins = [18, 24, 30, 40, 50, 60, 200]
etiquetas = ['18 a 24', '24 a 30', '30 a 40', '40 a 50', '50 a 60', '60 en adelante']

ft['GRUPO_EDAD'] = pd.cut(ft['P6040'], bins=bins, labels=etiquetas, right=False)
desempleados['GRUPO_EDAD'] = pd.cut(desempleados['P6040'], bins=bins, labels=etiquetas, right=False)

Ahora agrupamos con groupby por departamento y grupo de edad, y sumamos el factor de expansión FEX_C18. Lo hacemos con la PEA (FT == 1) y con los desempleados (DSI == 1). Después juntamos las dos series en una tabla y calculamos la tasa de desempleo.

Un detalle importante: si un grupo no tiene ningún desempleado, no aparece en la serie de desempleados y queda vacío (NaN) al juntar las tablas. Ese vacío en realidad es un cero, entonces lo rellenamos con fillna(0). Si no lo hiciéramos, esos grupos nunca saldrían como los de menor tasa.

In [0]:
# PEA y desempleados por departamento y grupo de edad
pea_edad = ft[ft['FT'] == 1].groupby(
    ['DPTO', 'GRUPO_EDAD'], observed=True)['FEX_C18'].sum()

desemp_edad = desempleados[desempleados['DSI'] == 1].groupby(
    ['DPTO', 'GRUPO_EDAD'], observed=True)['FEX_C18'].sum()

# Tabla con la tasa de desempleo
tabla_edad = pd.DataFrame({'PEA': pea_edad, 'DESEMPLEADOS': desemp_edad})
tabla_edad['DESEMPLEADOS'] = tabla_edad['DESEMPLEADOS'].fillna(0)
tabla_edad['TD'] = tabla_edad['DESEMPLEADOS'] / tabla_edad['PEA'] * 100

tabla_edad = tabla_edad.reset_index()
tabla_edad['GRUPO_EDAD'] = tabla_edad['GRUPO_EDAD'].astype(str)
tabla_edad['Departamento'] = tabla_edad['DPTO'].map(dpto_nombre)


Ahora nos quedamos solo con los grupos que pide el enunciado, descartando el de 50 a 60. Después, para cada departamento, buscamos con idxmax y idxmin la fila con la mayor y la menor tasa de desempleo, y juntamos los dos resultados en una sola tabla.

In [0]:
# Solo los grupos que pide el taller (se descarta 50 a 60)
tabla_edad = tabla_edad[tabla_edad['GRUPO_EDAD'] != '50 a 60']
tabla_edad = tabla_edad[['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']]

# Grupo con mayor y menor tasa por departamento
idx_max = tabla_edad.groupby('Departamento')['TD'].idxmax()
idx_min = tabla_edad.groupby('Departamento')['TD'].idxmin()

mayor = tabla_edad.loc[idx_max, ['Departamento', 'GRUPO_EDAD', 'TD']].rename(
    columns={'GRUPO_EDAD': 'Grupo mayor TD', 'TD': 'Mayor TD (%)'})
menor = tabla_edad.loc[idx_min, ['Departamento', 'GRUPO_EDAD', 'TD']].rename(
    columns={'GRUPO_EDAD': 'Grupo menor TD', 'TD': 'Menor TD (%)'})

resultado_edad = mayor.merge(menor, on='Departamento').sort_values('Departamento')
resultado_edad

_**R/ En casi todos los departamentos pasa lo mismo: el grupo de 18 a 24 años es el que tiene la mayor tasa de desempleo (en 25 de los 33 departamentos) y el de 60 años en adelante el que tiene la menor (en 24 de los 33).**_


### 4. ¿Cuáles son los 3 departamentos con menor tasa de desempleo de mujeres entre 
### los 18 y los 25 años ? 

Para este punto usamos la misma fórmula de siempre, TD = desempleadas / PEA × 100, pero solo con las mujeres de 18 a 25 años. Primero filtramos las dos tablas (ft y desempleados) con dos condiciones: sexo igual a 2 (mujer, variable P3271) y edad entre 18 y 25 (variable P6040). Usamos between(18, 25), que incluye las dos edades de los extremos.

In [0]:
# Filtro: mujeres entre 18 y 25 años (ambos extremos incluidos)
mujeres_ft = ft[
    (ft['P3271'] == 2) &
    (ft['P6040'].between(18, 25))
]

mujeres_desemp = desempleados[
    (desempleados['P3271'] == 2) &
    (desempleados['P6040'].between(18, 25))
]

Ahora agrupamos por departamento y sumamos el factor de expansión FEX_C18, para obtener la PEA y las desempleadas de cada uno. También contamos cuántas mujeres de la muestra hay en la PEA de cada departamento (CASOS_MUESTRA), porque en los departamentos con muy pocos casos la tasa es poco confiable. Con eso armamos la tabla y calculamos la tasa. Los departamentos sin ninguna desempleada quedan vacíos (NaN) y los rellenamos con 0.

In [0]:
# PEA, desempleadas y casos de muestra por departamento
pea_mujeres = mujeres_ft[mujeres_ft['FT'] == 1].groupby('DPTO')['FEX_C18'].sum()
desemp_mujeres = mujeres_desemp[mujeres_desemp['DSI'] == 1].groupby('DPTO')['FEX_C18'].sum()
casos_pea = mujeres_ft[mujeres_ft['FT'] == 1].groupby('DPTO').size()

# Tabla con la tasa de desempleo
tabla_mujeres = pd.DataFrame({
    'PEA': pea_mujeres,
    'DESEMPLEADAS': desemp_mujeres,
    'CASOS_MUESTRA': casos_pea
})
tabla_mujeres['DESEMPLEADAS'] = tabla_mujeres['DESEMPLEADAS'].fillna(0)
tabla_mujeres['TD'] = tabla_mujeres['DESEMPLEADAS'] / tabla_mujeres['PEA'] * 100

tabla_mujeres = tabla_mujeres.reset_index()
tabla_mujeres['Departamento'] = tabla_mujeres['DPTO'].map(dpto_nombre)
tabla_mujeres = tabla_mujeres[['Departamento', 'PEA', 'DESEMPLEADAS', 'CASOS_MUESTRA', 'TD']]

Por último ordenamos la tabla de menor a mayor tasa de desempleo y nos quedamos con los tres primeros.

In [0]:
tabla_mujeres.sort_values('TD')

In [0]:
# Los 3 departamentos con menor tasa de desempleo
tabla_mujeres.sort_values('TD', ascending=True).head(3)

_**R/ Los tres departamentos con menor tasa de desempleo entre las mujeres de 18 a 25 años a junio de 2026 son Vaupés (0 %), Nariño (4,29 %) y Amazonas (4,87 %).**_

### 5. Calcule el promedio de ingresos laborales (variable INGLABO) de las personas 
### residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle 
### del Cauca.

Para este punto usamos la tabla de ocupados, porque solo las personas que trabajan tienen ingreso laboral (INGLABO). Primero nos quedamos con los cinco departamentos que pide el enunciado, usando sus códigos: Antioquia (5), Caldas (17), Quindío (63), Risaralda (66) y Valle del Cauca (76).

In [0]:
# Verificar que las variables estén en la tabla de ocupados
print('INGLABO' in ocupados.columns, 'DPTO' in ocupados.columns)

dptos_interes = [5, 17, 63, 66, 76]
ocup_5 = ocupados[ocupados['DPTO'].isin(dptos_interes)].copy()

# Asegurar que sea numérica 
ocup_5['INGLABO'] = pd.to_numeric(ocup_5['INGLABO'], errors='coerce')

Ahora nos quedamos solo con las personas que tienen un ingreso laboral reportado, es decir, con INGLABO mayor que 0. Dejamos por fuera a quienes no reportaron ingreso o tienen ingreso cero (por ejemplo, trabajadores familiares sin remuneración), porque no queremos que bajen el promedio.

In [0]:
# Solo personas con ingreso laboral reportado
ocup_5 = ocup_5[ocup_5['INGLABO'].notna() & (ocup_5['INGLABO'] > 0)]

Para el promedio no sacamos un promedio simple, sino uno ponderado con el factor de expansión FEX_C18, para que el resultado represente a toda la población ocupada del departamento y no solo a las personas de la muestra. Entonces multiplicamos el ingreso de cada persona por su factor, sumamos por departamento, y dividimos entre la suma de los factores (el número de ocupados que representan).

Además dejamos la columna CASOS_MUESTRA, para ver cuántas personas de la encuesta hay en cada departamento.

In [0]:
# Ingreso ponderado: INGLABO x factor de expansión
ocup_5['ING_POND'] = ocup_5['INGLABO'] * ocup_5['FEX_C18']

g = ocup_5.groupby('DPTO')
tabla_ingresos = pd.DataFrame({
    'ING_TOTAL': g['ING_POND'].sum(),
    'OCUPADOS': g['FEX_C18'].sum(),
    'CASOS_MUESTRA': g.size()
})
tabla_ingresos['PROMEDIO_INGLABO'] = tabla_ingresos['ING_TOTAL'] / tabla_ingresos['OCUPADOS']

tabla_ingresos = tabla_ingresos.reset_index()
tabla_ingresos['Departamento'] = tabla_ingresos['DPTO'].map(dpto_nombre)
tabla_ingresos = tabla_ingresos[['Departamento', 'OCUPADOS', 'CASOS_MUESTRA', 'PROMEDIO_INGLABO']]
tabla_ingresos = tabla_ingresos.sort_values('PROMEDIO_INGLABO', ascending=False)

Por último armamos la tabla final con el promedio de cada departamento. Calculamos también el promedio de los cinco departamentos juntos y lo agregamos como una última fila, para tener todo en la misma tabla.

In [0]:
# Promedio conjunto de los 5 departamentos
prom_total = ocup_5['ING_POND'].sum() / ocup_5['FEX_C18'].sum()

# Fila con el total de los 5 departamentos
fila_total = pd.DataFrame([{
    'Departamento': 'TOTAL 5 DEPARTAMENTOS',
    'OCUPADOS': tabla_ingresos['OCUPADOS'].sum(),
    'CASOS_MUESTRA': tabla_ingresos['CASOS_MUESTRA'].sum(),
    'PROMEDIO_INGLABO': prom_total
}])

# Tabla final
tabla_final = pd.concat([tabla_ingresos, fila_total], ignore_index=True)
tabla_final['OCUPADOS'] = tabla_final['OCUPADOS'].round(0)
tabla_final['PROMEDIO_INGLABO'] = tabla_final['PROMEDIO_INGLABO'].round(0)

In [0]:
tabla_final

_**R/ El promedio de ingresos laborales mensuales a junio de 2026 es de $2,486,207 en Antioquia, $2,234,061 en Risaralda, $2,226,450 en Caldas, $2,171,560 en Quindío y $2,147,996 en el Valle del Cauca. El promedio de los cinco departamentos juntos es de $2,332,699.**_